# Flesch-Kincaid on ServiceNow

Download the publicly avilable ServiceNow technical documentation corpus, and run a traditional readability metric on the corpus. We use a simple Flesh-Kincaid score over the corpus.

We utilize the Python package TextStat to calculate these scores. See [TextStat documentation](https://pypi.org/project/textstat/) for explanation of the Flesch scores and other possible scores to use.

Note: Each of the notebooks re-downloads the corpus as a temporary folder on Google Colab. I did this to make each of the notebooks be self-contained.

# Download packages and data

In [ ]:
# Clone the Git repository. The specific branch 'australia' will be checked out.
# Note: To access a specific directory within a branch, you first clone the entire repository.
!git clone --branch australia https://github.com/ServiceNow/ServiceNowDocs.git

In [ ]:
!pip install textstat

In [ ]:
import os
import re
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
import textstat
import glob
from google.colab import files


# Helper to clean markdown formatting

In [ ]:
def clean_markdown(md_text: str) -> str:
    """Removes basic markdown syntax to ensure accurate text metrics."""
    # Remove headers, links, images, and code blocks
    text = re.sub(re.compile(r'```.*?```', re.DOTALL), '', md_text)
    text = re.sub(re.compile(r'`.*?`'), '', text)
    text = re.sub(r'#+\s+', '', text)
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text)
    text = re.sub(r'[*_~`\-+>]', '', text)
    return text.strip()


# Worker function for a single file

In [ ]:
def process_single_file(file_path: str) -> dict:
    """Reads, cleans, and analyzes a single file."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            raw_content = f.read()

        cleaned_text = clean_markdown(raw_content)

        # Skip completely empty files
        if not cleaned_text.strip():
            return {"file_name": os.path.basename(file_path), "error": "Empty file"}

        return {
            "file_name": file_path,
            "reading_ease": textstat.flesch_reading_ease(cleaned_text),
            "grade_level": textstat.flesch_kincaid_grade(cleaned_text),
            "word_count": textstat.lexicon_count(cleaned_text),
            "error": None
        }
    except Exception as e:
        return {"file_name": os.path.basename(file_path), "error": str(e)}



# Extract scores from the entire corpus

In [ ]:
def process_corpus(corpus_dir: str, output_csv: str) -> None:
    """Finds all md files, runs parallel processing, and saves data to CSV."""
    print("Gathering markdown files...")
    file_paths   = glob.glob(os.path.join(corpus_dir, "**", "*.md"), recursive=True)

    total_files = len(file_paths)
    print(f"Found {total_files:,} files. Starting parallel analysis...")

    # Run processing using all available CPU cores
    results = []
    with ProcessPoolExecutor() as executor:
        # map preserves the file order and acts as a generator
        for idx, result in enumerate(executor.map(process_single_file, file_paths), 1):
            results.append(result)

            # Print progress update every 2,000 files
            if idx % 2000 == 0 or idx == total_files:
                print(f"Processed {idx:,} / {total_files:,} files...")

    # Save results to a CSV file using Pandas
    print("Saving results to CSV...")
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Complete! Results saved to: {output_csv}")
    files.download(output_csv)


In [ ]:
# --- Execution ---
CORPUS_DIRECTORY = "/content/ServiceNowDocs/markdown/"
OUTPUT_FILE = "corpus_fleschkincaid_results.csv"

process_corpus(CORPUS_DIRECTORY, OUTPUT_FILE)